In [1]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().parent.resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(PROJECT_ROOT),
    )

from src.inference.predict import (
    SleepStagePredictor,
)

**Load final predictor**

In [2]:
predictor = SleepStagePredictor(
    model_path=(
        PROJECT_ROOT
        / "final_model"
        / "stacked_lstm_32_32.keras"
    ),

    scaler_path=(
        PROJECT_ROOT
        / "final_model"
        / "scaler.pkl"
    ),

    config_path=(
        PROJECT_ROOT
        / "final_model"
        / "config.json"
    ),
)

print(
    "Final predictor loaded successfully."
)

Final predictor loaded successfully.


**Load test features**

In [7]:
FEATURES_PATH = (
    PROJECT_ROOT
    / "data"
    / "features"
    / "sleep_edf_features.csv"
)

SPLIT_PATH = (
    PROJECT_ROOT
    / "data"
    / "features"
    / "subject_split.csv"
)

df = pd.read_csv(
    FEATURES_PATH
)

split_df = pd.read_csv(
    SPLIT_PATH
)

df = df.merge(
    split_df,
    on="subject_id",
    how="inner",
)

**Build the test sequences**

In [8]:
from src.preprocessing.normalization import (
    SleepFeatureScaler,
)

from src.preprocessing.sequences import (
    build_sequences_from_dataframe,
)

train_df = (
    df[
        df["split"] == "train"
    ]
    .sort_values(
        ["subject_id", "start_time"]
    )
    .reset_index(drop=True)
)

test_df = (
    df[
        df["split"] == "test"
    ]
    .sort_values(
        ["subject_id", "start_time"]
    )
    .reset_index(drop=True)
)

scaler = SleepFeatureScaler()

X_train_scaled = scaler.fit_transform(
    train_df
)

X_test_scaled = scaler.transform(
    test_df
)

X_test_final, y_test_labels = (
    build_sequences_from_dataframe(
        dataframe=test_df,
        feature_values=X_test_scaled,
        sequence_length=20,
    )
)

print(
    X_test_final.shape
)

(67017, 20, 10)


**Predict the first test sequence**

In [7]:
from src.inference.predict import (
    FEATURE_COLUMNS
)

raw_test_sequence = (
    test_df[
        FEATURE_COLUMNS
    ]
    .iloc[:20]
    .copy()
)

print(
    "Raw sequence shape:",
    raw_test_sequence.shape,
)

print(
    "\nFirst row of raw features:"
)

display(
    raw_test_sequence.head(1)
)

result = predictor.predict_sequence(
    raw_test_sequence
)

print(
    "\nPrediction:"
)

print(
    result
)

Raw sequence shape: (20, 10)

First row of raw features:


,delta_absolute,theta_absolute,alpha_absolute,beta_absolute,delta_relative,theta_relative,alpha_relative,beta_relative,spectral_entropy,dominant_frequency
0,2.063891e-10,2.072891e-11,6.479465e-12,8.046230e-12,0.854105,0.085783,0.026814,0.033298,0.667242,0.585938



Prediction:
{'predicted_stage': 'W', 'predicted_index': 0, 'probabilities': {'W': 0.999996542930603, 'N1': 2.3999764380278066e-06, 'N2': 7.816909715074871e-07, 'N3': 3.5433731504497246e-09, 'REM': 2.1721105269989494e-07}}


**Batch prediction test**

In [8]:
predictions, probabilities = (
    predictor.predict_batch(
        X_test_final[:100]
    )
)

print(
    "Predictions shape:",
    predictions.shape,
)

print(
    "Probabilities shape:",
    probabilities.shape,
)

print(
    "First 10 predictions:"
)

print(
    predictions[:10]
)

Predictions shape: (100,)
Probabilities shape: (100, 5)
First 10 predictions:
[0 0 0 0 0 0 0 0 0 0]


**Batch prediction using ALREADY SCALED sequences**

In [9]:
predictions, probabilities = (
    predictor.predict_batch(
        X_test_final[:100]
    )
)

print(
    "Predictions shape:",
    predictions.shape,
)

print(
    "Probabilities shape:",
    probabilities.shape,
)

Predictions shape: (100,)
Probabilities shape: (100, 5)


**Verify the inference API**

In [10]:
from src.inference.predict import (
    SleepStagePredictor,
)

In [11]:
predictor = SleepStagePredictor(
    model_path=(
        PROJECT_ROOT
        / "final_model"
        / "stacked_lstm_32_32.keras"
    ),
    scaler_path=(
        PROJECT_ROOT
        / "final_model"
        / "scaler.pkl"
    ),
    config_path=(
        PROJECT_ROOT
        / "final_model"
        / "config.json"
    ),
)

print(
    "Predictor loaded successfully."
)

Predictor loaded successfully.


In [12]:
import numpy as np

raw_sequences = np.stack(
    [
        test_df[
            FEATURE_COLUMNS
        ]
        .iloc[
            i:i + 20
        ]
        .to_numpy(
            dtype=np.float32
        )
        for i in [0, 20, 40]
    ],
    axis=0,
)

print(
    "Raw batch shape:",
    raw_sequences.shape,
)

Raw batch shape: (3, 20, 10)


In [ ]:
predictions, probabilities = (
    predictor.predict_batch(
        raw_sequences
    )
)

print(
    "Predictions:",
    predictions
)

print(
    "Predictions shape:",
    predictions.shape
)

print(
    "Probabilities shape:",
    probabilities.shape
)

NameError: name 'raw_sequences' is not defined

In [10]:
import numpy as np
from src.inference.predict import (
    FEATURE_COLUMNS
)

raw_sequences = np.stack(
    [
        test_df[
            FEATURE_COLUMNS
        ]
        .iloc[
            i:i + 20
        ]
        .to_numpy(
            dtype=np.float32
        )
        for i in [0, 20, 40]
    ],
    axis=0,
)

print(
    "Raw batch shape:",
    raw_sequences.shape,
)

Raw batch shape: (3, 20, 10)


In [11]:
predictions, probabilities = (
    predictor.predict_batch(
        raw_sequences
    )
)

print(
    "Predictions:",
    predictions
)

print(
    "Predictions shape:",
    predictions.shape
)

print(
    "Probabilities shape:",
    probabilities.shape
)

Predictions: [0 0 0]
Predictions shape: (3,)
Probabilities shape: (3, 5)


# Temporary notebook cells


**Verify recording ID → participant ID mapping**

In [19]:
def extract_participant_id(recording_id: str) -> str:
    """
    Convert a Sleep-EDF recording ID to its participant ID.

    Examples:
        SC4001 → SC400
        SC4002 → SC400
        SC4011 → SC401
        ST7011 → ST701
        ST7012 → ST701
    """

    recording_id = str(
        recording_id
    ).strip()

    if len(recording_id) != 6:
        raise ValueError(
            f"Unexpected recording ID: "
            f"{recording_id}"
        )

    return recording_id[:5]


df["participant_id"] = (
    df["subject_id"]
    .map(
        extract_participant_id
    )
)

print(
    df[
        [
            "subject_id",
            "participant_id",
        ]
    ]
    .drop_duplicates()
    .tail(20)
)

       subject_id participant_id
438258     ST7141          ST714
439116     ST7142          ST714
439989     ST7151          ST715
440880     ST7152          ST715
441921     ST7161          ST716
442978     ST7162          ST716
443966     ST7171          ST717
444927     ST7172          ST717
445927     ST7181          ST718
446886     ST7182          ST718
447853     ST7191          ST719
448808     ST7192          ST719
449739     ST7201          ST720
450660     ST7202          ST720
451557     ST7211          ST721
452634     ST7212          ST721
453644     ST7221          ST722
454673     ST7222          ST722
455623     ST7241          ST724
456638     ST7242          ST724


**See how many real participants we have**

In [20]:
print(
    "Recording IDs:",
    df["subject_id"].nunique(),
)

print(
    "Participant IDs:",
    df["participant_id"].nunique(),
)

print(
    "\nRecordings per participant:"
)

print(
    df[
        [
            "participant_id",
            "subject_id",
        ]
    ]
    .drop_duplicates()
    .groupby(
        "participant_id"
    )
    .size()
    .value_counts()
    .sort_index()
)

Recording IDs: 197
Participant IDs: 100

Recordings per participant:
1     3
2    97
Name: count, dtype: int64


**Check participant overlap across the CURRENT recording-level split**

In [21]:
current_split = (
    df[
        [
            "participant_id",
            "subject_id",
            "split",
        ]
    ]
    .drop_duplicates()
)

participant_split_counts = (
    current_split
    .groupby(
        "participant_id"
    )["split"]
    .nunique()
)

leaked_participants = (
    participant_split_counts[
        participant_split_counts > 1
    ]
)

print(
    "Participants appearing in multiple splits:",
    len(leaked_participants),
)

print(
    "\nLeaked participants:"
)

display(
    current_split[
        current_split[
            "participant_id"
        ].isin(
            leaked_participants.index
        )
    ]
    .sort_values(
        [
            "participant_id",
            "split",
        ]
    )
)

Participants appearing in multiple splits: 43

Leaked participants:


,participant_id,subject_id,split
0,SC400,SC4001,train
2650,SC400,SC4002,val
16688,SC403,SC4031,test
19508,SC403,SC4032,train
27597,SC405,SC4051,train
...,...,...,...
440880,ST715,ST7152,val
450660,ST720,ST7202,test
449739,ST720,ST7201,train
453644,ST722,ST7221,test


**Create participant-level train/validation/test split**

In [22]:
from sklearn.model_selection import train_test_split

# One row per participant.
participant_df = (
    df[
        [
            "participant_id",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        "participant_id"
    )
    .reset_index(drop=True)
)

print(
    "Total participants:",
    len(participant_df),
)

# Keep SC/ST proportions approximately balanced.
participant_df["dataset_type"] = (
    participant_df["participant_id"]
    .str[:2]
)

train_parts, temp_parts = train_test_split(
    participant_df,
    test_size=0.30,
    random_state=42,
    stratify=participant_df["dataset_type"],
)

val_parts, test_parts = train_test_split(
    temp_parts,
    test_size=0.50,
    random_state=42,
    stratify=temp_parts["dataset_type"],
)

train_parts = train_parts.copy()
val_parts = val_parts.copy()
test_parts = test_parts.copy()

train_parts["split"] = "train"
val_parts["split"] = "val"
test_parts["split"] = "test"

participant_split_df = pd.concat(
    [
        train_parts,
        val_parts,
        test_parts,
    ],
    ignore_index=True,
)

print(
    "\nParticipant counts:"
)

print(
    participant_split_df[
        "split"
    ].value_counts()
)

print(
    "\nDataset type by split:"
)

print(
    pd.crosstab(
        participant_split_df[
            "split"
        ],
        participant_split_df[
            "dataset_type"
        ],
    )
)

Total participants: 100

Participant counts:
split
train    70
val      15
test     15
Name: count, dtype: int64

Dataset type by split:
dataset_type  SC  ST
split               
test          12   3
train         55  15
val           11   4


**Verify participant-level split integrity**

In [23]:
train_subjects = set(
    participant_split_df[
        participant_split_df["split"] == "train"
    ]["participant_id"]
)

val_subjects = set(
    participant_split_df[
        participant_split_df["split"] == "val"
    ]["participant_id"]
)

test_subjects = set(
    participant_split_df[
        participant_split_df["split"] == "test"
    ]["participant_id"]
)

print(
    "Train ∩ Validation:",
    train_subjects & val_subjects,
)

print(
    "Train ∩ Test:",
    train_subjects & test_subjects,
)

print(
    "Validation ∩ Test:",
    val_subjects & test_subjects,
)

print(
    "\nAll participants mutually exclusive:",
    (
        not (
            train_subjects & val_subjects
            or
            train_subjects & test_subjects
            or
            val_subjects & test_subjects
        )
    ),
)

Train ∩ Validation: set()
Train ∩ Test: set()
Validation ∩ Test: set()

All participants mutually exclusive: True


**Apply participant-level split to recordings**

In [24]:
df_participant_split = df.drop(
    columns=[
        "split"
    ],
    errors="ignore",
).merge(
    participant_split_df[
        [
            "participant_id",
            "split",
        ]
    ],
    on="participant_id",
    how="left",
    validate="many_to_one",
)

print(
    "Missing split assignments:",
    df_participant_split[
        "split"
    ].isna().sum(),
)

print(
    "\nRecording counts by split:"
)

print(
    df_participant_split[
        [
            "subject_id",
            "participant_id",
            "split",
        ]
    ]
    .drop_duplicates()
    [
        "split"
    ]
    .value_counts()
)

print(
    "\nParticipant counts by split:"
)

print(
    df_participant_split[
        [
            "participant_id",
            "split",
        ]
    ]
    .drop_duplicates()
    [
        "split"
    ]
    .value_counts()
)

Missing split assignments: 0

Recording counts by split:
split
train    138
test      30
val       29
Name: count, dtype: int64

Participant counts by split:
split
train    70
val      15
test     15
Name: count, dtype: int64


**Save corrected participant split**

In [25]:
PARTICIPANT_SPLIT_PATH = (
    PROJECT_ROOT
    / "data"
    / "features"
    / "participant_split.csv"
)

participant_split_df[
    [
        "participant_id",
        "dataset_type",
        "split",
    ]
].sort_values(
    [
        "split",
        "participant_id",
    ]
).to_csv(
    PARTICIPANT_SPLIT_PATH,
    index=False,
)

print(
    "Corrected participant split saved to:"
)

print(
    PARTICIPANT_SPLIT_PATH
)

Corrected participant split saved to:
D:\Local Disk\Uni Sources\Term 8\Final Project\Sleep-Stage-Prediction\data\features\participant_split.csv


**Save recording → participant → split mapping**

In [26]:
RECORDING_MAPPING_PATH = (
    PROJECT_ROOT
    / "data"
    / "features"
    / "recording_participant_split.csv"
)

(
    df_participant_split[
        [
            "subject_id",
            "participant_id",
            "split",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "participant_id",
            "subject_id",
        ]
    )
    .to_csv(
        RECORDING_MAPPING_PATH,
        index=False,
    )
)

print(
    "Recording mapping saved to:"
)

print(
    RECORDING_MAPPING_PATH
)

Recording mapping saved to:
D:\Local Disk\Uni Sources\Term 8\Final Project\Sleep-Stage-Prediction\data\features\recording_participant_split.csv


**Confirm no participant leakage**

In [27]:
check_df = (
    df_participant_split[
        [
            "participant_id",
            "split",
        ]
    ]
    .drop_duplicates()
)

split_counts = (
    check_df
    .groupby(
        "participant_id"
    )["split"]
    .nunique()
)

leaked = split_counts[
    split_counts > 1
]

print(
    "Participants appearing in multiple splits:",
    len(leaked),
)

assert len(leaked) == 0

print(
    "Participant leakage check: PASSED"
)

Participants appearing in multiple splits: 0
Participant leakage check: PASSED
